# DexBot on Google Colab

Run the full DexBot app in this Colab notebook. It downloads the project, installs dependencies, starts the FastAPI server, and embeds the UI below.

**Run all cells** (`Runtime` → `Run all`) and wait a few seconds.

### Requirements

Nothing needs to be installed manually. The notebook downloads the public repository, installs its dependencies, starts the server, and embeds the UI.

If you copy this notebook into a private fork, add a fine-grained GitHub token with **Contents: Read-only** access as a Colab secret named `GITHUB_TOKEN`.

> API keys are optional. DexBot starts in Demo mode with no keys. Add provider keys to `.env` in the project folder and restart the server to use live models.


In [ ]:
import io
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

REPO_DIR = Path("/content/dexbot")
REPO_SLUG = "unknownboiz/dexor"
BRANCH = "main"
PORT = 8000


def github_token() -> str:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata

        return (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        return ""


def download_project() -> None:
    url = f"https://api.github.com/repos/{REPO_SLUG}/zipball/{BRANCH}"
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    token = github_token()
    if token:
        request.add_header("Authorization", f"Bearer {token}")

    try:
        with urllib.request.urlopen(request, timeout=90) as response:
            payload = response.read()
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            raise RuntimeError(
                "Could not download the DexBot repository. If you are using a private "
                "fork, add a GitHub token with Contents: Read-only access as a "
                "Colab secret named GITHUB_TOKEN."
            ) from None
        raise

    target = Path("/content/.dexbot-download")
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir(parents=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(target)
        extracted = next(target.iterdir())

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.move(str(extracted), str(REPO_DIR))
    shutil.rmtree(target, ignore_errors=True)
    print(f"Downloaded {REPO_SLUG} to {REPO_DIR}")


if not (REPO_DIR / "run.py").exists():
    download_project()
else:
    print(f"Using existing project at {REPO_DIR}")

print("Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)


def dexbot_ready() -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=3) as response:
            return json.load(response).get("app") == "DexBot"
    except Exception:
        return False


if not dexbot_ready():
    print("Starting DexBot...")
    with open("/tmp/dexbot.log", "w") as log:
        subprocess.Popen(
            [sys.executable, "run.py"],
            cwd=REPO_DIR,
            stdout=log,
            stderr=subprocess.STDOUT,
            env={**os.environ, "PORT": str(PORT)},
            start_new_session=True,
        )

    for _ in range(60):
        if dexbot_ready():
            break
        time.sleep(0.5)
    else:
        log_text = Path("/tmp/dexbot.log").read_text(errors="replace")
        raise RuntimeError("DexBot did not start.\n\n" + log_text[-4000:])
else:
    print("DexBot is already running.")

print(f"DexBot is ready at http://localhost:{PORT}")

try:
    from google.colab import output

    output.serve_kernel_port_as_iframe(PORT, height="900")
except ImportError:
    print("Open the URL above in your browser.")


## Optional: add API keys

DexBot works immediately in **Demo mode**. To use a live provider:

1. In the Colab file browser (folder icon), open `/content/dexbot/.env.example`.
2. Copy it to `/content/dexbot/.env`.
3. Add the key for your provider, for example `XAI_API_KEY=...` or `OPENAI_API_KEY=...`.
4. Run the restart cell below.

See `README.md` in the project folder for all supported providers and settings.


In [ ]:
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO_DIR = Path("/content/dexbot")
PORT = 8000

subprocess.run(["pkill", "-f", "[r]un.py"], check=False)
time.sleep(1)

with open("/tmp/dexbot.log", "w") as log:
    subprocess.Popen(
        [sys.executable, "run.py"],
        cwd=REPO_DIR,
        stdout=log,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PORT": str(PORT)},
        start_new_session=True,
    )

for _ in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=2) as response:
            if response.status == 200:
                print(f"DexBot restarted at http://localhost:{PORT}")
                break
    except Exception:
        pass
    time.sleep(0.5)
else:
    print(Path("/tmp/dexbot.log").read_text(errors="replace")[-4000:])
    raise RuntimeError("DexBot did not restart")
